In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EKSTRAKSI 5000 SAMPEL ACAK DARI STEAD (DENGAN PENANGANAN NOISE)
- Untuk EV: gunakan p_arrival_sample dari atribut
- Untuk NO: ambil 7 detik pertama sebagai sinyal, 7 detik berikutnya sebagai noise
- Output JSON 1C dan 3C
"""

import os
import json
import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm
from datetime import datetime

# =============================================
# 1. KONFIGURASI
# =============================================
CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
OUTPUT_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'

SAMPLE_RATE = 100.0
SIG_DURATION = 7.0
NOISE_DURATION = 7.0
NORM_WINDOW = 9.0
TARGET_LEN = int(SAMPLE_RATE * SIG_DURATION)  # 700

N_SAMPLES = 5000
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================
# 2. BACA CSV (SEMUA KOLOM)
# =============================================
print("="*80)
print("📂 MEMBACA METADATA STEAD (CSV)")
print("="*80)

try:
    df = pd.read_csv(CSV_PATH, low_memory=False)
except Exception:
    df = pd.read_csv(CSV_PATH, engine='python')

print(f"✅ Total rows: {len(df):,}")

# Pilih kolom yang diperlukan
needed_cols = [
    'trace_name', 'trace_category',
    'source_origin_time', 'source_magnitude',
    'source_latitude', 'source_longitude',
    'network_code', 'receiver_code',
    'snr_db'
]
available_cols = [col for col in needed_cols if col in df.columns]
df = df[available_cols]

# =============================================
# 3. FILTER
# =============================================
print("\n" + "="*80)
print("🔍 FILTER DATA")
print("="*80)

df = df[df['trace_category'].isin(['earthquake_local', 'noise'])]
print(f"✅ Setelah filter trace_category: {len(df):,}")

if len(df) == 0:
    print("❌ Tidak ada data. Periksa CSV.")
    exit()

if len(df) < N_SAMPLES:
    N_SAMPLES = len(df)
    print(f"⚠️ Menggunakan semua data ({N_SAMPLES})")

# =============================================
# 4. SAMPLING ACAK SEIMBANG
# =============================================
print("\n" + "="*80)
print("🎲 MEMILIH SAMPEL ACAK SEIMBANG")
print("="*80)

df_ev = df[df['trace_category'] == 'earthquake_local']
df_no = df[df['trace_category'] == 'noise']
print(f"Earthquake: {len(df_ev):,}, Noise: {len(df_no):,}")

target_ev = N_SAMPLES // 2
target_no = N_SAMPLES - target_ev

if len(df_ev) < target_ev:
    target_ev = len(df_ev)
    target_no = N_SAMPLES - target_ev
if len(df_no) < target_no:
    target_no = len(df_no)
    target_ev = N_SAMPLES - target_no

selected_ev = df_ev.sample(n=target_ev, random_state=RANDOM_SEED) if target_ev > 0 else pd.DataFrame()
selected_no = df_no.sample(n=target_no, random_state=RANDOM_SEED) if target_no > 0 else pd.DataFrame()
selected_df = pd.concat([selected_ev, selected_no]).sample(frac=1, random_state=RANDOM_SEED)
print(f"✅ Total selected: {len(selected_df)}")
print(f"   EV: {len(selected_df[selected_df['trace_category'] == 'earthquake_local'])}")
print(f"   NO: {len(selected_df[selected_df['trace_category'] == 'noise'])}")

# =============================================
# 5. BUKA HDF5
# =============================================
print("\n" + "="*80)
print("📊 MEMBUKA HDF5")
print("="*80)

hf = h5py.File(HDF5_PATH, 'r')
print(f"Keys di root: {list(hf.keys())}")

# =============================================
# 6. FUNGSI EKSTRAKSI (DENGAN PENANGANAN NOISE)
# =============================================
def extract_windows_ev(waveform, p_idx):
    """Ekstrak sinyal dan noise untuk event gempa menggunakan P-arrival."""
    p_idx = int(p_idx)
    wf = waveform.T  # (3, 6000)
    wf = np.array([wf[2, :], wf[1, :], wf[0, :]])  # Z, N, E
    
    sig_start = p_idx
    sig_end = p_idx + TARGET_LEN
    noise_start = p_idx - TARGET_LEN
    noise_end = p_idx
    
    if sig_end > wf.shape[1]: sig_end = wf.shape[1]
    if noise_start < 0: noise_start = 0
    
    signal = wf[:, sig_start:sig_end]
    noise = wf[:, noise_start:noise_end]
    
    if signal.shape[1] < TARGET_LEN:
        pad = ((0,0), (0, TARGET_LEN - signal.shape[1]))
        signal = np.pad(signal, pad, 'constant')
    if noise.shape[1] < TARGET_LEN:
        pad = ((0,0), (0, TARGET_LEN - noise.shape[1]))
        noise = np.pad(noise, pad, 'constant')
    
    # Normalisasi
    norm_start = p_idx
    norm_end = p_idx + int(NORM_WINDOW * SAMPLE_RATE)
    if norm_end > wf.shape[1]: norm_end = wf.shape[1]
    if norm_end > norm_start:
        norm_win = wf[:, norm_start:norm_end]
        max_val = np.max(np.abs(norm_win))
    else:
        max_val = np.max(np.abs(signal))
    if np.isnan(max_val) or max_val < 1e-6:
        max_val = 1.0
    
    signal_norm = signal / max_val
    noise_norm = noise / max_val
    
    return (signal_norm[0, :].tolist(), signal_norm[1, :].tolist(), signal_norm[2, :].tolist(),
            noise_norm[0, :].tolist(), noise_norm[1, :].tolist(), noise_norm[2, :].tolist())

def extract_windows_noise(waveform):
    """
    Ekstrak sinyal dan noise untuk noise.
    Sinyal: 7 detik pertama (0-700), Noise: 7 detik berikutnya (700-1400).
    """
    wf = waveform.T  # (3, 6000)
    wf = np.array([wf[2, :], wf[1, :], wf[0, :]])  # Z, N, E
    
    # Sinyal: 0-700
    signal = wf[:, 0:TARGET_LEN]
    # Noise: 700-1400
    noise = wf[:, TARGET_LEN:2*TARGET_LEN]
    
    # Padding jika kurang
    if signal.shape[1] < TARGET_LEN:
        pad = ((0,0), (0, TARGET_LEN - signal.shape[1]))
        signal = np.pad(signal, pad, 'constant')
    if noise.shape[1] < TARGET_LEN:
        pad = ((0,0), (0, TARGET_LEN - noise.shape[1]))
        noise = np.pad(noise, pad, 'constant')
    
    # Normalisasi dengan max absolut dari sinyal
    max_val = np.max(np.abs(signal))
    if np.isnan(max_val) or max_val < 1e-6:
        max_val = 1.0
    
    signal_norm = signal / max_val
    noise_norm = noise / max_val
    
    return (signal_norm[0, :].tolist(), signal_norm[1, :].tolist(), signal_norm[2, :].tolist(),
            noise_norm[0, :].tolist(), noise_norm[1, :].tolist(), noise_norm[2, :].tolist())

# =============================================
# 7. EKSTRAKSI PER SAMPEL
# =============================================
print("\n" + "="*80)
print("⏳ EKSTRAKSI SINYAL & NOISE (700 SAMPAI)")
print("="*80)

data_1c = {}
data_3c = {}
failed = 0

for idx, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="Ekstraksi"):
    trace_name = row['trace_name']
    category = row.get('trace_category', 'unknown')
    
    try:
        dataset = hf['data/' + trace_name]
    except KeyError:
        failed += 1
        continue
    
    waveform = np.array(dataset)
    
    # ===== EKSTRAKSI UNTUK GEMPA =====
    if category == 'earthquake_local':
        if 'p_arrival_sample' not in dataset.attrs:
            failed += 1
            continue
        try:
            p_arr = int(dataset.attrs['p_arrival_sample'])
        except:
            failed += 1
            continue
        if p_arr < 0 or p_arr >= 6000:
            failed += 1
            continue
        
        try:
            Z, N, E, Z_noise, N_noise, E_noise = extract_windows_ev(waveform, p_arr)
        except Exception:
            failed += 1
            continue
        
        s_arr = dataset.attrs.get('s_arrival_sample', -1)
        if isinstance(s_arr, (np.integer, int)):
            s_arr = int(s_arr)
        else:
            s_arr = -1
        
        metadata = {
            'trace_name': trace_name,
            'trace_category': category,
            'p_arrival_sample': p_arr,
            's_arrival_sample': s_arr,
            'source_origin_time': row.get('source_origin_time', ''),
            'source_magnitude': row.get('source_magnitude', np.nan),
            'source_latitude': row.get('source_latitude', np.nan),
            'source_longitude': row.get('source_longitude', np.nan),
            'network_code': row.get('network_code', ''),
            'receiver_code': row.get('receiver_code', ''),
            'snr_db': row.get('snr_db', np.nan)
        }
        cat = 'ev'
    
    # ===== EKSTRAKSI UNTUK NOISE =====
    else:
        try:
            Z, N, E, Z_noise, N_noise, E_noise = extract_windows_noise(waveform)
        except Exception:
            failed += 1
            continue
        
        metadata = {
            'trace_name': trace_name,
            'trace_category': category,
            'p_arrival_sample': -1,
            's_arrival_sample': -1,
            'source_origin_time': row.get('source_origin_time', ''),
            'source_magnitude': np.nan,
            'source_latitude': row.get('source_latitude', np.nan),
            'source_longitude': row.get('source_longitude', np.nan),
            'network_code': row.get('network_code', ''),
            'receiver_code': row.get('receiver_code', ''),
            'snr_db': row.get('snr_db', np.nan)
        }
        cat = 'no'
    
    data_1c[trace_name] = {
        'type': cat,
        'Z': Z,
        'Z_noise': Z_noise,
        'metadata': metadata
    }
    data_3c[trace_name] = {
        'type': cat,
        'Z': Z,
        'N': N,
        'E': E,
        'Z_noise': Z_noise,
        'N_noise': N_noise,
        'E_noise': E_noise,
        'metadata': metadata
    }

hf.close()

# =============================================
# 8. SIMPAN JSON
# =============================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_1c_path = os.path.join(OUTPUT_DIR, f"STEAD_5000_1C_{timestamp}.json")
json_3c_path = os.path.join(OUTPUT_DIR, f"STEAD_5000_3C_{timestamp}.json")

with open(json_1c_path, 'w') as f:
    json.dump(data_1c, f, indent=2)
with open(json_3c_path, 'w') as f:
    json.dump(data_3c, f, indent=2)

print(f"\n✅ 1C saved: {json_1c_path}")
print(f"✅ 3C saved: {json_3c_path}")

print(f"\n📊 Statistik:")
print(f"Total: {len(data_1c)}")
cats = [v['type'] for v in data_1c.values()]
print(f"EV: {cats.count('ev')}, NO: {cats.count('no')}")
print(f"❌ Gagal: {failed}")

if len(data_1c) > 0:
    sample_key = list(data_1c.keys())[0]
    print(f"\nSample key: {sample_key}")
    print(f"Z len: {len(data_1c[sample_key]['Z'])}")
    print(f"Z_noise len: {len(data_1c[sample_key]['Z_noise'])}")

📂 MEMBACA METADATA STEAD (CSV)
✅ Total rows: 1,265,657

🔍 FILTER DATA
✅ Setelah filter trace_category: 1,265,657

🎲 MEMILIH SAMPEL ACAK SEIMBANG
Earthquake: 1,030,231, Noise: 235,426
✅ Total selected: 5000
   EV: 2500
   NO: 2500

📊 MEMBUKA HDF5
Keys di root: ['data']

⏳ EKSTRAKSI SINYAL & NOISE (700 SAMPAI)


Ekstraksi: 100%|██████████| 5000/5000 [00:37<00:00, 132.60it/s]



✅ 1C saved: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json
✅ 3C saved: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_3C_20260719_062844.json

📊 Statistik:
Total: 5000
EV: 2500, NO: 2500
❌ Gagal: 0

Sample key: O03E.TA_20141107191307_EV
Z len: 700
Z_noise len: 700


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VALIDASI JSON STEAD 5000
"""

import json
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

JSON_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json'
OUTPUT_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/validasi'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================
# 1. LOAD JSON
# =============================================
print("="*80)
print("📂 MEMUAT JSON STEAD 5000")
print("="*80)

with open(JSON_PATH, 'r') as f:
    data = json.load(f)

print(f"✅ Total entri: {len(data)}")

# =============================================
# 2. STATISTIK UMUM
# =============================================
print("\n" + "="*80)
print("📊 STATISTIK UMUM")
print("="*80)

keys = list(data.keys())
sample_key = keys[0]
sample = data[sample_key]

print(f"Sample key: {sample_key}")
print(f"Keys dalam entri: {list(sample.keys())}")
print(f"Metadata keys: {list(sample['metadata'].keys())}")
print(f"Panjang Z: {len(sample['Z'])}")
print(f"Panjang Z_noise: {len(sample['Z_noise'])}")

# Hitung kelas
classes = [v['type'] for v in data.values()]
ev_count = classes.count('ev')
no_count = classes.count('no')
print(f"\nDistribusi kelas:")
print(f"  EV: {ev_count}")
print(f"  NO: {no_count}")
print(f"  Total: {len(data)}")

# =============================================
# 3. VALIDASI PANJANG ARRAY
# =============================================
print("\n" + "="*80)
print("📏 VALIDASI PANJANG ARRAY")
print("="*80)

z_lengths = [len(v['Z']) for v in data.values()]
z_noise_lengths = [len(v['Z_noise']) for v in data.values()]

unique_z = set(z_lengths)
unique_z_noise = set(z_noise_lengths)

print(f"Panjang Z unik: {unique_z}")
print(f"Panjang Z_noise unik: {unique_z_noise}")

if len(unique_z) == 1 and list(unique_z)[0] == 700:
    print("✅ Semua Z panjangnya 700")
else:
    print("⚠️ Ada variasi panjang Z")

if len(unique_z_noise) == 1 and list(unique_z_noise)[0] == 700:
    print("✅ Semua Z_noise panjangnya 700")
else:
    print("⚠️ Ada variasi panjang Z_noise")

# =============================================
# 4. VALIDASI NILAI SINYAL (NaN, Inf, Rentang)
# =============================================
print("\n" + "="*80)
print("🔍 VALIDASI NILAI SINYAL")
print("="*80)

has_nan = 0
has_inf = 0
min_val = float('inf')
max_val = -float('inf')

for key, record in data.items():
    z = np.array(record['Z'])
    if np.isnan(z).any():
        has_nan += 1
    if np.isinf(z).any():
        has_inf += 1
    min_val = min(min_val, np.min(z))
    max_val = max(max_val, np.max(z))

print(f"Entri dengan NaN: {has_nan}")
print(f"Entri dengan Inf: {has_inf}")
print(f"Nilai minimum global: {min_val:.6f}")
print(f"Nilai maksimum global: {max_val:.6f}")

if has_nan == 0 and has_inf == 0:
    print("✅ Tidak ada NaN atau Inf")
else:
    print("⚠️ Ada NaN atau Inf! Periksa ulang.")

if min_val >= -1.0 and max_val <= 1.0:
    print("✅ Nilai dalam rentang [-1, 1] (normalisasi OK)")
else:
    print(f"⚠️ Nilai di luar rentang: min={min_val}, max={max_val}")

# =============================================
# 5. VALIDASI METADATA (p_arrival_sample)
# =============================================
print("\n" + "="*80)
print("⏰ VALIDASI METADATA p_arrival_sample")
print("="*80)

ev_has_p = 0
no_has_p = 0
ev_missing_p = 0
no_missing_p = 0

for key, record in data.items():
    cat = record['type']
    p_val = record['metadata'].get('p_arrival_sample', None)
    if cat == 'ev':
        if p_val is not None and p_val >= 0:
            ev_has_p += 1
        else:
            ev_missing_p += 1
    else:  # no
        if p_val is not None and p_val >= 0:
            no_has_p += 1
        else:
            no_missing_p += 1

print(f"EV - memiliki p_arrival_sample: {ev_has_p}")
print(f"EV - missing p_arrival_sample: {ev_missing_p}")
print(f"NO - memiliki p_arrival_sample: {no_has_p}")
print(f"NO - missing p_arrival_sample: {no_missing_p}")

if ev_missing_p == 0 and no_has_p == 0:
    print("✅ Metadata p_arrival_sample konsisten (EV punya, NO tidak)")
else:
    print("⚠️ Ada inkonsistensi metadata p_arrival_sample")

# =============================================
# 6. VISUALISASI SAMPEL ACAK
# =============================================
print("\n" + "="*80)
print("🖼️ VISUALISASI SAMPEL ACAK")
print("="*80)

# Pilih 4 sampel acak (2 EV, 2 NO)
import random
random.seed(42)
keys_list = list(data.keys())
ev_keys = [k for k in keys_list if data[k]['type'] == 'ev']
no_keys = [k for k in keys_list if data[k]['type'] == 'no']

selected_ev = random.sample(ev_keys, min(2, len(ev_keys)))
selected_no = random.sample(no_keys, min(2, len(no_keys)))
selected_keys = selected_ev + selected_no

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
fig.suptitle('Contoh Sinyal Z dan Noise dari STEAD 5000', fontsize=16)

for i, key in enumerate(selected_keys):
    record = data[key]
    z = np.array(record['Z'])
    z_noise = np.array(record['Z_noise'])
    cat = record['type']
    p = record['metadata'].get('p_arrival_sample', -1)
    
    ax1 = axes[i, 0]
    ax1.plot(z, color='blue', alpha=0.8)
    ax1.set_title(f"{key[:20]}... ({cat}) | P={p}")
    ax1.set_xlabel('Sampel')
    ax1.set_ylabel('Amplitudo')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-1.2, 1.2)
    
    ax2 = axes[i, 1]
    ax2.plot(z_noise, color='orange', alpha=0.8)
    ax2.set_title(f"Noise window")
    ax2.set_xlabel('Sampel')
    ax2.set_ylabel('Amplitudo')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(-1.2, 1.2)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'sample_waveforms.png')
plt.savefig(plot_path, dpi=150)
print(f"✅ Plot tersimpan: {plot_path}")
plt.close()

# =============================================
# 7. RINGKASAN VALIDASI
# =============================================
print("\n" + "="*80)
print("✅ RINGKASAN VALIDASI")
print("="*80)

print(f"Total entri: {len(data)}")
print(f"EV: {ev_count}, NO: {no_count}")
print(f"Panjang Z: {list(unique_z)}")
print(f"Panjang Z_noise: {list(unique_z_noise)}")
print(f"NaN/Inf: {has_nan} / {has_inf}")
print(f"Rentang nilai: [{min_val:.4f}, {max_val:.4f}]")
print(f"Metadata p_arrival: EV={ev_has_p}/{ev_count}, NO={no_missing_p}/{no_count}")

if (len(unique_z) == 1 and list(unique_z)[0] == 700 and
    len(unique_z_noise) == 1 and list(unique_z_noise)[0] == 700 and
    has_nan == 0 and has_inf == 0 and
    min_val >= -1.0 and max_val <= 1.0 and
    ev_missing_p == 0 and no_has_p == 0):
    print("\n🎉 SEMUA VALIDASI LULUS!")
    print("✅ JSON STEAD 5000 SIAP DIGUNAKAN UNTUK BENCHMARKING")
else:
    print("\n⚠️ Ada masalah dalam dataset. Periksa detail di atas.")

print("="*80)

📂 MEMUAT JSON STEAD 5000
✅ Total entri: 5000

📊 STATISTIK UMUM
Sample key: O03E.TA_20141107191307_EV
Keys dalam entri: ['type', 'Z', 'Z_noise', 'metadata']
Metadata keys: ['trace_name', 'trace_category', 'p_arrival_sample', 's_arrival_sample', 'source_origin_time', 'source_magnitude', 'source_latitude', 'source_longitude', 'network_code', 'receiver_code', 'snr_db']
Panjang Z: 700
Panjang Z_noise: 700

Distribusi kelas:
  EV: 2500
  NO: 2500
  Total: 5000

📏 VALIDASI PANJANG ARRAY
Panjang Z unik: {700}
Panjang Z_noise unik: {700}
✅ Semua Z panjangnya 700
✅ Semua Z_noise panjangnya 700

🔍 VALIDASI NILAI SINYAL
Entri dengan NaN: 0
Entri dengan Inf: 0
Nilai minimum global: -1.000000
Nilai maksimum global: 1.000000
✅ Tidak ada NaN atau Inf
✅ Nilai dalam rentang [-1, 1] (normalisasi OK)

⏰ VALIDASI METADATA p_arrival_sample
EV - memiliki p_arrival_sample: 2500
EV - missing p_arrival_sample: 0
NO - memiliki p_arrival_sample: 0
NO - missing p_arrival_sample: 2500
✅ Metadata p_arrival_sample ko